# Khám phá Olist thật trước khi xây nền tảng

Trước khi viết DDL hay DAG, Data Engineer phải biết dữ liệu **thật sự** trông thế nào: khoá có trùng
không, ngày nào thiếu, bảng nào nhân dòng khi join, con số kinh doanh nào đáng đưa lên dashboard. Notebook
này làm việc đó trên bộ **Brazilian E-Commerce Public Dataset by Olist** thật (Kaggle, CC BY-NC-SA 4.0):
~99 nghìn đơn 09/2016–10/2018, 9 tệp CSV ~120 MB trong `data/raw/`.

Việc của bạn, theo thứ tự một pipeline:

1. **làm sạch**: `clean_orders` (cờ giao hàng, độ trễ, mốc thời gian vô lý), `dedup_reviews` (một review mỗi đơn);
2. **bảng sạch mức đơn**: `build_order_facts`, mỗi đơn đúng một dòng, tiền không bị nhân khi join;
3. **phân tích**: trễ hạn vs điểm review, trễ theo bang, phương thức thanh toán, độ tập trung seller,
   doanh thu theo danh mục, tỉ lệ khách mua lại; gom thành `insights`;
4. **ghi dữ liệu sạch**: `write_clean` ra `data/clean/olist/order_facts.parquet`, chạy lại không nhân đôi;
5. vẽ biểu đồ và viết `report.md` cạnh notebook.

Kết quả của notebook là đầu vào cho M01 (DDL, kiểm tra số dòng), M02 (grain, `is_late`), M09 (KPI dashboard)
và M10 (rule chất lượng). Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu và vẽ; khi chấm chúng
bị bỏ qua, test gọi thẳng các hàm bạn viết. Hàm còn `NotImplementedError` thì test của hàm đó **skip** kèm lời
nhắc, làm xong thì test chạy thật. Chấm:
`uv run pytest tracks/data-engineer/capstone/project/tests/test_olist_exploration.py -q -rs`

In [ ]:
# explore: tìm gốc repo (thư mục có pyproject.toml), tải Olist thật nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw"
CLEAN = ROOT / "data" / "clean" / "olist"

if not (RAW / "olist_orders_dataset.csv").exists():
    # Cần Kaggle token, xem đầu tệp data/download_olist.py. Không có token: dùng bộ giả lập
    # (uv run python data/generate_olist.py --out data/raw/olist_synthetic) và đổi RAW sang thư mục đó.
    from data.download_olist import DATASET, EXPECTED, ensure_kaggle_credentials, kaggle_download

    if not ensure_kaggle_credentials():
        raise SystemExit("Chưa có Kaggle token: xem hướng dẫn ở đầu data/download_olist.py")
    kaggle_download(DATASET, RAW, EXPECTED)
print(RAW, sorted(p.name for p in RAW.glob("*.csv")))

In [ ]:
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd

FILES = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "translation": "product_category_name_translation.csv",
}
DATES = {
    "orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
    "items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}
# Zip là mã, không phải số: đọc kiểu chuỗi để giữ số 0 đầu ("01310", không phải 1310).
STRINGS = {"customer_zip_code_prefix": str, "seller_zip_code_prefix": str}
EXCLUDED_STATUSES = ("canceled", "unavailable")   # không tính là một lần mua thành công
DELAY_BUCKETS = ["sớm ≥ 7 ngày", "sớm 1–6 ngày", "đúng ngày", "trễ 1–7 ngày", "trễ > 7 ngày"]
FACT_COLUMNS = [
    "order_id", "customer_unique_id", "customer_state", "order_status", "order_purchase_timestamp",
    "purchase_month", "order_estimated_delivery_date", "order_delivered_customer_date",
    "is_delivered", "delivery_days", "delay_days", "is_late", "bad_timeline",
    "n_items", "items_value", "freight_value", "payment_value", "review_score",
]


def load_olist(raw_dir: Path) -> dict[str, pd.DataFrame]:
    """Đọc 8 tệp CSV Olist (bỏ geolocation, 1 triệu dòng tọa độ chưa cần) thành dict tên -> DataFrame.

    Viết sẵn. Cột ngày đã là datetime, zip là chuỗi.
    """
    raw_dir = Path(raw_dir)
    return {
        name: pd.read_csv(raw_dir / filename, parse_dates=DATES.get(name, []), dtype=STRINGS)
        for name, filename in FILES.items()
    }

In [ ]:
# explore: nạp và soi nhanh: số dòng, khoá có trùng không, cột nào thiếu
tables = load_olist(RAW)
for name, df in tables.items():
    nulls = df.isna().sum()
    print(f"{name:12} {len(df):>8,} dòng | thiếu: {nulls[nulls > 0].to_dict()}")
print("review_id trùng:", tables["reviews"]["review_id"].duplicated().sum(),
      "| đơn có > 1 review:", tables["reviews"]["order_id"].duplicated().sum())
print(tables["orders"]["order_status"].value_counts())

## 1. Làm sạch đơn hàng

Bộ thật có những chỗ bộ giả lập không có: 8 đơn `delivered` thiếu ngày giao, vài trăm đơn có
`order_delivered_carrier_date` **trước** ngày mua, ngày giao dự kiến luôn là 00:00. `clean_orders(orders)`
trả bản mới (không sửa `orders`), **giữ đủ mọi dòng**, thêm các cột:

| Cột | Quy tắc |
| --- | --- |
| `is_delivered` | `order_status == "delivered"` **và** có `order_delivered_customer_date` |
| `delivery_days` | (giao − mua) tính bằng ngày, số thực; `NaN` nếu không `is_delivered` |
| `delay_days` | **ngày lịch** giao − **ngày lịch** dự kiến (bỏ giờ, `.dt.normalize()`), kiểu `Int64`; `<NA>` nếu không `is_delivered`. Dương là trễ |
| `is_late` | `delay_days > 0`, kiểu `boolean` (nullable): giao trong ngày dự kiến **không** trễ; `<NA>` nếu chưa giao |
| `purchase_month` | `"YYYY-MM"` của `order_purchase_timestamp` |
| `bad_timeline` | `True` nếu giao cho hãng vận chuyển trước lúc mua, hoặc khách nhận trước lúc giao hãng. Gắn cờ, **không xoá** |

Vì sao không xoá dòng bất thường: xoá im lặng làm lệch số đơn so với nguồn (M01 kiểm số dòng), và
dòng lạ thường là manh mối lỗi hệ thống nguồn. Gắn cờ để downstream tự quyết.

In [ ]:
def clean_orders(orders: pd.DataFrame) -> pd.DataFrame:
    """Bản sao của orders, đủ dòng, thêm is_delivered, delivery_days, delay_days, is_late, purchase_month, bad_timeline."""
    out = orders.copy()
    delivered_at = out["order_delivered_customer_date"]
    out["is_delivered"] = (out["order_status"] == "delivered") & delivered_at.notna()
    days = (delivered_at - out["order_purchase_timestamp"]).dt.total_seconds() / 86400
    out["delivery_days"] = days.where(out["is_delivered"])
    # So ngày lịch: dự kiến luôn là 00:00, so cả giờ thì đơn giao 15h đúng ngày hẹn cũng thành "trễ".
    delay = (delivered_at.dt.normalize() - out["order_estimated_delivery_date"].dt.normalize()).dt.days
    out["delay_days"] = delay.where(out["is_delivered"]).astype("Int64")
    out["is_late"] = (out["delay_days"] > 0).astype("boolean")
    out["purchase_month"] = out["order_purchase_timestamp"].dt.strftime("%Y-%m")
    carrier = out["order_delivered_carrier_date"]
    out["bad_timeline"] = (carrier < out["order_purchase_timestamp"]) | (delivered_at < carrier)
    return out

`dedup_reviews(reviews)` trả **một dòng mỗi `order_id`** với cột `order_id`, `review_id`, `review_score`,
`review_answer_timestamp`. Một đơn có nhiều review thì giữ review **trả lời sau cùng**
(`review_answer_timestamp` lớn nhất; hoà thì `review_creation_date`, rồi `review_id` lớn hơn). Bộ thật có
~550 đơn có hơn một review và ~800 `review_id` dùng lại cho nhiều đơn: join thẳng `orders` với `reviews`
sẽ nhân dòng.

In [ ]:
def dedup_reviews(reviews: pd.DataFrame) -> pd.DataFrame:
    """Một dòng mỗi order_id: review trả lời sau cùng. Cột order_id, review_id, review_score, review_answer_timestamp."""
    ordered = reviews.sort_values(["order_id", "review_answer_timestamp", "review_creation_date", "review_id"])
    last = ordered.drop_duplicates("order_id", keep="last")
    return last[["order_id", "review_id", "review_score", "review_answer_timestamp"]].reset_index(drop=True)

## 2. Bảng sạch mức đơn: `build_order_facts`

`build_order_facts(tables)` nhận dict của `load_olist`, trả DataFrame **mỗi đơn đúng một dòng**, đúng các
cột trong `FACT_COLUMNS` theo thứ tự đó:

- cột đơn hàng từ `clean_orders`; `customer_unique_id`, `customer_state` từ `customers`
  (`customer_id` là id **của đơn**, `customer_unique_id` mới là người mua);
- `n_items`, `items_value` (tổng `price`), `freight_value` (tổng `freight_value`) từ `items`;
  `payment_value` (tổng) từ `payments`: **gom theo `order_id` trước rồi mới join**, nếu join thẳng thì một
  đơn 2 item × 2 payment thành 4 dòng và tiền bị nhân đôi;
- `review_score` (kiểu `Int64`) từ `dedup_reviews`;
- đơn không có item/payment (đơn huỷ, `unavailable`) có `n_items = 0` và tiền `0.0`, không có review thì
  `review_score` là `<NA>`.

Mẹo: `merge(..., validate="one_to_one")` làm pandas báo lỗi ngay khi bạn lỡ join nhân dòng.

In [ ]:
def build_order_facts(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Mỗi đơn một dòng, cột FACT_COLUMNS. Gom items/payments theo order_id trước khi join."""
    orders = clean_orders(tables["orders"])
    items = tables["items"].groupby("order_id").agg(
        n_items=("order_item_id", "size"), items_value=("price", "sum"), freight_value=("freight_value", "sum"))
    payments = tables["payments"].groupby("order_id").agg(payment_value=("payment_value", "sum"))
    reviews = dedup_reviews(tables["reviews"]).set_index("order_id")[["review_score"]]
    customers = tables["customers"][["customer_id", "customer_unique_id", "customer_state"]]
    facts = (
        orders.merge(customers, on="customer_id", how="left", validate="one_to_one")
        .merge(items, left_on="order_id", right_index=True, how="left", validate="one_to_one")
        .merge(payments, left_on="order_id", right_index=True, how="left", validate="one_to_one")
        .merge(reviews, left_on="order_id", right_index=True, how="left", validate="one_to_one")
    )
    facts["n_items"] = facts["n_items"].fillna(0).astype("int64")
    for col in ("items_value", "freight_value", "payment_value"):
        facts[col] = facts[col].fillna(0.0)
    facts["review_score"] = facts["review_score"].astype("Int64")
    return facts[FACT_COLUMNS].reset_index(drop=True)

In [ ]:
# explore: bảng sạch mức đơn
facts = build_order_facts(tables)
facts.info()
assert facts["order_id"].is_unique and len(facts) == len(tables["orders"])
print("tổng items_value:", round(facts["items_value"].sum(), 2), "| tổng price gốc:", round(tables["items"]["price"].sum(), 2))
facts.head()

## 3. Phân tích

Mỗi hàm trả một con số hoặc một bảng nhỏ, để vừa vẽ được vừa chấm được:

| Hàm | Trả về |
| --- | --- |
| `review_by_delay(facts)` | DataFrame index là `DELAY_BUCKETS` theo đúng thứ tự (`pd.cut` với mốc `[-inf, -7, -1, 0, 7, inf]`: ≤ −7, −6..−1, 0, 1..7, > 7), cột `orders`, `avg_review`, `low_review_share` (tỉ lệ review 1–2 sao). Chỉ đơn `is_delivered` có review |
| `late_share_by_state(facts, min_orders=100)` | DataFrame index `customer_state`, cột `delivered` (số đơn đã giao), `late_share` (float); bỏ bang có ít hơn `min_orders` đơn đã giao; sắp `late_share` giảm dần |
| `payment_mix(payments)` | DataFrame index `payment_type`, cột `orders` (số đơn **khác nhau** dùng phương thức đó), `value` (tổng tiền), `order_share` (`orders` / số đơn khác nhau trong `payments`; tổng > 1 vì một đơn có thể trả nhiều cách), `value_share`; sắp `orders` giảm dần |
| `seller_concentration(items, top=0.10)` | tỉ trọng GMV (tổng `price`) của `ceil(top × số seller)` seller bán nhiều nhất, float 0–1 |
| `category_revenue(items, products, translation)` | Series doanh thu (tổng `price`) theo tên danh mục **tiếng Anh**, sắp giảm dần. Danh mục không có bản dịch giữ tên gốc tiếng Bồ; sản phẩm không có danh mục là `"unknown"` |
| `repeat_customer_rate(facts)` | tỉ lệ `customer_unique_id` có ≥ 2 đơn, bỏ đơn có trạng thái trong `EXCLUDED_STATUSES` |

In [ ]:
def review_by_delay(facts: pd.DataFrame) -> pd.DataFrame:
    """Index DELAY_BUCKETS; cột orders, avg_review, low_review_share (đơn đã giao có review)."""
    rows = facts[facts["is_delivered"] & facts["review_score"].notna()]
    bucket = pd.cut(rows["delay_days"].astype(float), [-np.inf, -7, -1, 0, 7, np.inf], labels=DELAY_BUCKETS)
    score = rows["review_score"].astype(float)
    return (
        pd.DataFrame({"bucket": bucket, "score": score, "low": score <= 2})
        .groupby("bucket", observed=False)
        .agg(orders=("score", "size"), avg_review=("score", "mean"), low_review_share=("low", "mean"))
    )


def late_share_by_state(facts: pd.DataFrame, min_orders: int = 100) -> pd.DataFrame:
    """Index customer_state; cột delivered, late_share; bang >= min_orders đơn đã giao; late_share giảm dần."""
    rows = facts[facts["is_delivered"]]
    out = rows.assign(late=rows["is_late"].astype(float)).groupby("customer_state").agg(
        delivered=("order_id", "size"), late_share=("late", "mean"))
    out = out[out["delivered"] >= min_orders]
    return out.sort_values(["late_share", "delivered"], ascending=[False, False])


def payment_mix(payments: pd.DataFrame) -> pd.DataFrame:
    """Index payment_type; cột orders, value, order_share, value_share; orders giảm dần."""
    out = payments.groupby("payment_type").agg(orders=("order_id", "nunique"), value=("payment_value", "sum"))
    out["order_share"] = out["orders"] / payments["order_id"].nunique()
    out["value_share"] = out["value"] / out["value"].sum()
    return out.sort_values("orders", ascending=False)


def seller_concentration(items: pd.DataFrame, top: float = 0.10) -> float:
    """Tỉ trọng GMV (tổng price) của ceil(top * số seller) seller lớn nhất."""
    gmv = items.groupby("seller_id")["price"].sum().sort_values(ascending=False)
    k = math.ceil(len(gmv) * top)
    return float(gmv.iloc[:k].sum() / gmv.sum())


def category_revenue(items: pd.DataFrame, products: pd.DataFrame, translation: pd.DataFrame) -> pd.Series:
    """Tổng price theo danh mục tiếng Anh (thiếu bản dịch: tên gốc; thiếu danh mục: "unknown"), giảm dần."""
    english = dict(zip(translation["product_category_name"], translation["product_category_name_english"]))
    name = products["product_category_name"]
    category = pd.Series(name.map(english).fillna(name).fillna("unknown").values, index=products["product_id"])
    by_item = items["product_id"].map(category).fillna("unknown")
    return items.groupby(by_item.rename("category"))["price"].sum().sort_values(ascending=False)


def repeat_customer_rate(facts: pd.DataFrame) -> float:
    """Tỉ lệ customer_unique_id có >= 2 đơn, bỏ đơn có trạng thái trong EXCLUDED_STATUSES."""
    kept = facts[~facts["order_status"].isin(EXCLUDED_STATUSES)]
    per_customer = kept.groupby("customer_unique_id")["order_id"].nunique()
    return float((per_customer >= 2).mean())

In [ ]:
# explore: các bảng phân tích
display(review_by_delay(facts))
display(late_share_by_state(facts).head(10))
display(payment_mix(tables["payments"]))
display(category_revenue(tables["items"], tables["products"], tables["translation"]).head(10))

`insights(facts, tables, min_orders=100)` gom các con số cho báo cáo và dashboard, `dict` với đúng các khoá:

| Khoá | Nghĩa |
| --- | --- |
| `orders`, `delivered` | số đơn, số đơn `is_delivered` |
| `late_share` | tỉ lệ `is_late` trong các đơn `is_delivered` |
| `avg_review_on_time`, `avg_review_late` | điểm review trung bình của đơn đã giao có review, không trễ / trễ |
| `low_review_share_on_time`, `low_review_share_late` | tỉ lệ review 1–2 sao, cùng hai nhóm trên |
| `median_delivery_days` | trung vị `delivery_days` của đơn đã giao |
| `worst_state`, `worst_state_late_share` | dòng đầu của `late_share_by_state(facts, min_orders)` |
| `top_payment_type`, `top_payment_order_share` | dòng đầu của `payment_mix` |
| `card_installments_share` | tỉ lệ **dòng** payment `credit_card` có `payment_installments > 1` (trả góp) |
| `top10_seller_gmv_share` | `seller_concentration(items)` |
| `top_category`, `top_category_revenue` | dòng đầu của `category_revenue` |
| `repeat_customer_rate` | `repeat_customer_rate(facts)` |

In [ ]:
def insights(facts: pd.DataFrame, tables: dict[str, pd.DataFrame], min_orders: int = 100) -> dict:
    """Các con số trong bảng trên."""
    delivered = facts[facts["is_delivered"]]
    reviewed = delivered[delivered["review_score"].notna()]
    late = reviewed["is_late"].astype(bool)
    score = reviewed["review_score"].astype(float)
    states = late_share_by_state(facts, min_orders)
    payments = tables["payments"]
    mix = payment_mix(payments)
    card = payments[payments["payment_type"] == "credit_card"]
    categories = category_revenue(tables["items"], tables["products"], tables["translation"])
    return {
        "orders": len(facts),
        "delivered": int(facts["is_delivered"].sum()),
        "late_share": float(delivered["is_late"].astype(float).mean()),
        "avg_review_on_time": float(score[~late].mean()),
        "avg_review_late": float(score[late].mean()),
        "low_review_share_on_time": float((score[~late] <= 2).mean()),
        "low_review_share_late": float((score[late] <= 2).mean()),
        "median_delivery_days": float(delivered["delivery_days"].median()),
        "worst_state": states.index[0],
        "worst_state_late_share": float(states["late_share"].iloc[0]),
        "top_payment_type": mix.index[0],
        "top_payment_order_share": float(mix["order_share"].iloc[0]),
        "card_installments_share": float((card["payment_installments"] > 1).mean()),
        "top10_seller_gmv_share": seller_concentration(tables["items"]),
        "top_category": categories.index[0],
        "top_category_revenue": float(categories.iloc[0]),
        "repeat_customer_rate": repeat_customer_rate(facts),
    }

In [ ]:
# explore: các con số để viết report.md
INSIGHTS = insights(facts, tables)
INSIGHTS

## 4. Biểu đồ

Vẽ ít nhất hai biểu đồ (mỗi biểu đồ một cell `explore`):

1. điểm review trung bình và tỉ lệ 1–2 sao theo nhóm trễ (`review_by_delay`): trễ bao nhiêu ngày thì khách bắt đầu giận?
2. tỉ lệ giao trễ theo bang (`late_share_by_state`): miền nào giao kém, có trùng khoảng cách tới São Paulo (nơi phần lớn seller ở)?
3. (tuỳ chọn) số đơn và GMV theo `purchase_month`: tháng nào vọt (Black Friday 11/2017), tháng nào dữ liệu bị cắt cụt?

In [ ]:
# explore: review theo độ trễ
import matplotlib.pyplot as plt

by_delay = review_by_delay(facts)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
by_delay["avg_review"].plot.bar(ax=axes[0], color="#4c72b0", title="Điểm review trung bình")
(by_delay["low_review_share"] * 100).plot.bar(ax=axes[1], color="#c44e52", title="% review 1–2 sao")
for ax in axes:
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()

In [ ]:
# explore: giao trễ theo bang
states = late_share_by_state(facts)
ax = (states["late_share"] * 100).sort_values().plot.barh(figsize=(7, 8), color="#dd8452")
ax.axvline(INSIGHTS["late_share"] * 100, color="black", linestyle="--", label="cả nước")
ax.set_xlabel("% đơn giao trễ")
ax.legend()
plt.tight_layout()

In [ ]:
# explore: đơn và GMV theo tháng
monthly = facts.groupby("purchase_month").agg(orders=("order_id", "size"), gmv=("items_value", "sum"))
fig, ax = plt.subplots(figsize=(12, 4))
monthly["orders"].plot(ax=ax, marker="o", label="số đơn")
ax2 = ax.twinx()
(monthly["gmv"] / 1e6).plot(ax=ax2, color="#55a868", label="GMV (triệu BRL)")
ax.set_xlabel("")
ax.set_title("11/2017: Black Friday; 09–10/2018: dữ liệu bị cắt")
plt.tight_layout()

## 5. Ghi dữ liệu sạch

`write_clean(facts, out_dir)` ghi **một** tệp `out_dir/order_facts.parquet` (ghi đè, tạo thư mục nếu chưa có)
và trả đường dẫn. Ghi ra tệp tạm trong cùng thư mục rồi `os.replace` để không bao giờ để lại tệp dở. Chạy
notebook hai lần thì `data/clean/olist/` vẫn đúng số dòng: **idempotent**. `data/clean/` không commit.

In [ ]:
def write_clean(facts: pd.DataFrame, out_dir: Path) -> Path:
    """Ghi đè out_dir/order_facts.parquet (qua tệp tạm), trả đường dẫn."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    target = out_dir / "order_facts.parquet"
    tmp = out_dir / f".{target.name}.{os.getpid()}.tmp"   # cùng thư mục: os.replace mới nguyên tử
    try:
        facts.to_parquet(tmp, index=False)
        os.replace(tmp, target)
    finally:
        tmp.unlink(missing_ok=True)
    return target

In [ ]:
# explore: ghi data/clean/olist/order_facts.parquet
path = write_clean(facts, CLEAN)
print(path, f"{path.stat().st_size / 1e6:.1f} MB", len(pd.read_parquet(path)), "dòng")

## 6. Báo cáo

Viết `report.md` cạnh notebook này, đủ 4 mục `## Bộ dữ liệu`, `## Làm sạch`, `## Insight`, `## Giới hạn`.
Insight phải có số (lấy từ `INSIGHTS`), mỗi con số kèm một câu "nghĩa là gì cho nền tảng / cho kinh doanh".
Những con số này là KPI bạn sẽ đưa lên dashboard M09 và đối chiếu khi kiểm thử pipeline.